In [92]:
from copy import deepcopy
from tqdm import tqdm
from helpers import get_TBlogger, Settings
from models import DataModel

import glob
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn import metrics
import numpy as np

from utils import load_models

import logging

logger = logging.getLogger(__name__)

settings = Settings()

TBlogger = get_TBlogger(settings.LOGGING_DIR, settings.EXPERIMENT_NAME)


In [93]:
dataModel = DataModel(settings.DATASET_DIR, 
                    settings.CSV_FILE, 
                    settings.FUNDUS_DIR, 
                    settings.TRAIN_SIZE, 
                    settings.INCLUDE_TEST)

In [94]:




trasnformations =A.Compose([
    A.RandomResizedCrop((224, 224), scale=(0.9, 1.0)),
    #A.HorizontalFlip(p=0.5),
    #A.Rotate(limit=10, p=0.5),
    #A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.7),
    #A.ColorJitter(
    #     brightness=0.1,
    #     contrast=0.1,
    #     saturation=0.05,
    #     hue=0.01,
    #     p=0.5
    # ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

train_loader, val_loader, test_loader = dataModel.get_loaders(settings.BATCH_SIZE, trasnformations)



if settings.USE_ALL_EXPERIMENTS_TO_PREDICT:
    experiments = glob.glob("../logs/*")
else:
    experiments = [f"../logs/{settings.PREDICT_EXPERIMENT_NAME}"]

for PREDICT_EXPERIMENT_NAME in experiments:
    print("#"*40, "Experiment Name:", PREDICT_EXPERIMENT_NAME.split("/")[-1], "#"*40)

    model_lst, passed_models = load_models(modelList=["resnet152"], pretrained=True, EXPERIMENT_NAME=PREDICT_EXPERIMENT_NAME)
    
    for model_name, model in model_lst:
        if model_name not in passed_models:
            with torch.inference_mode():
                model.to("cuda")
                model.eval()
                logits_list= []
                label_list = []
                prob_list = []
                for i, l in val_loader: 
                    l= l.long()
                    logits= model.model(i.to("cuda"))
                    logits_list = np.concatenate((logits_list, logits.argmax(axis=1).detach().cpu().numpy()), axis=None)
                    
                    logits = torch.sigmoid(logits)[:, 1]                  
                    label_list = np.concatenate((label_list, l), axis=None)
                    prob_list = np.concatenate((prob_list, logits.detach().cpu().numpy()), axis=None)
                
                fpr, tpr, thresholds = metrics.roc_curve(label_list, prob_list)

                # print(f"{"#"*20} Model Name: {model_name} {"#"*20}\nTrue predictions: {TP},\nTotal predictions: {TOTP},\nAccuracy: {(TP/TOTP)}%")
                print()
                print()






######################################## Experiment Name: nighnth_try_pretrained ########################################




In [95]:
thres_val = round(thresholds[np.argmax(tpr - fpr)], 3)
thres_val

np.float64(0.444)

In [96]:
thres_val = round(thresholds[np.argmax(tpr - fpr)], 3)
prob_list[np.where(prob_list >= thres_val)] = 1
prob_list[np.where(prob_list != 1)] = 0

In [97]:
tn, fp, fn, tp = metrics.confusion_matrix(label_list, prob_list).ravel()
print(f'AUROC on the validation set is {round(metrics.auc(fpr, tpr), 3)}')
print(f'Threshold determined by Youden index is {thres_val}')
print(f'Accuracy on the validation set is {round((tp + tn) / (tp + tn + fp + fn), 3)}')
print(f'Sensitivity on the validation set is {round(tp / (tp + fn), 3)}')
print(f'Specificity on the validation set is {round(tn / (tn + fp), 3)}')

AUROC on the validation set is 0.981
Threshold determined by Youden index is 0.444
Accuracy on the validation set is 0.934
Sensitivity on the validation set is 0.927
Specificity on the validation set is 0.938


In [99]:
prob_list = []
label_list = []
with torch.no_grad():
    for data in tqdm(test_loader):
        images, labels = data[0].float().to("cuda"), data[1].cpu().numpy()
        outputs = model(images)
        label_list = np.concatenate((label_list, labels), axis=None)
        prob_list = np.concatenate((prob_list, torch.sigmoid(outputs)[:, 1].detach().cpu().numpy()), axis=None)

100%|██████████| 116/116 [01:15<00:00,  1.53it/s]


In [103]:
fpr, tpr, thresholds = metrics.roc_curve(label_list, prob_list)
precision, recall, thresholds = metrics.precision_recall_curve(label_list, prob_list)
auc_std, prc_std, acc_std, sen_std, spe_std, ppv_std, npv_std = \
    bootstrap_cls(prob_list=prob_list, label_list=label_list, threshold=thres_val, times=100)

prob_list[np.where(prob_list >= thres_val)] = 1
prob_list[np.where(prob_list != 1)] = 0
tn, fp, fn, tp = metrics.confusion_matrix(label_list, prob_list).ravel()

print(f'AUROC on the test set is {round(metrics.auc(fpr, tpr), 3)}')
print(f'AUPRC on the test set is {round(metrics.auc(recall, precision), 3)}')
print(f'Threshold determined by Youden index is {thres_val}')
print(f'Accuracy on the test set is {round((tp + tn) / (tp + tn + fp + fn), 3)}')
print(f'Sensitivity on the test set is {round(tp / (tp + fn), 3)}')
print(f'Specificity on the test set is {round(tn / (tn + fp), 3)}')
print(f'PPV on the test set is {round(tp / (tp + fp), 3)}')
print(f'NPV on the test set is {round(tn / (tn + fn), 3)}')

AUROC on the test set is 0.987
AUPRC on the test set is 0.981
Threshold determined by Youden index is 0.444
Accuracy on the test set is 0.949
Sensitivity on the test set is 0.938
Specificity on the test set is 0.956
PPV on the test set is 0.931
NPV on the test set is 0.961


In [104]:
results_df= []
results_df.append([f"DeiT-Tiny", f"{id}", f"{thres_val}",
                       f"{format(metrics.auc(fpr, tpr), '.3f')} ({auc_std})",
                       f"{format(metrics.auc(recall, precision), '.3f')} ({prc_std})",
                       f"{format((tp + tn) / (tp + tn + fp + fn), '.3f')} ({acc_std})",
                       f"{format(tp / (tp + fn), '.3f')} ({sen_std})",
                       f"{format(tn / (tn + fp), '.3f')} ({spe_std})",
                       f"{format(tp / (tp + fp), '.3f')} ({ppv_std})",
                       f"{format(tn / (tn + fn), '.3f')} ({npv_std})",
                       ])

In [105]:
import pandas as pd 

In [106]:
results_df = pd.DataFrame(results_df)
results_df.columns = ['Model', 'Data Split', 'Threshold', 'AUROC', 'AUPRC', 'Accuracy', 'Sensitivity', 'Specificity',
                      'PPV', 'NPV']
# results_df.to_csv("results_deit_tiny_cls.csv", index=False, encoding="cp1252")

In [107]:
results_df

,Model,Data Split,Threshold,AUROC,AUPRC,Accuracy,Sensitivity,Specificity,PPV,NPV
0,DeiT-Tiny,<built-in function id>,0.444,0.987 (0.002184551566301814),0.981 (0.002892627751206244),0.949 (0.004642543829630602),0.938 (0.009039487332412608),0.956 (0.005055365943367392),0.931 (0.00800353374026536),0.961 (0.005986319711326544)


In [102]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    confusion_matrix
)

def bootstrap_cls(prob_list, label_list, threshold=0.5, times=100, random_state=None):
    rng = np.random.default_rng(random_state)
    
    prob_list = np.asarray(prob_list)
    label_list = np.asarray(label_list)
    n = len(label_list)
    
    # Store metrics across bootstrap samples
    aucs, prcs = [], []
    accs, sens, spes = [], [], []
    ppvs, npvs = [], []
    
    for _ in range(times):
        # Sample with replacement
        indices = rng.integers(0, n, n)
        probs_sample = prob_list[indices]
        labels_sample = label_list[indices]
        
        # Convert probabilities to predicted labels
        preds = (probs_sample >= threshold).astype(int)
        
        # --- Metrics ---
        # AUC (handle edge case: only one class present)
        try:
            auc = roc_auc_score(labels_sample, probs_sample)
        except ValueError:
            auc = np.nan
        
        # PRC AUC
        try:
            prc = average_precision_score(labels_sample, probs_sample)
        except ValueError:
            prc = np.nan
        
        # Accuracy
        acc = accuracy_score(labels_sample, preds)
        
        # Confusion matrix: tn, fp, fn, tp
        tn, fp, fn, tp = confusion_matrix(labels_sample, preds, labels=[0, 1]).ravel()
        
        # Sensitivity (Recall)
        sen = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        
        # Specificity
        spe = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        
        # PPV (Precision)
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        
        # NPV
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
        
        # Collect
        aucs.append(auc)
        prcs.append(prc)
        accs.append(acc)
        sens.append(sen)
        spes.append(spe)
        ppvs.append(ppv)
        npvs.append(npv)
    
    # Convert to arrays and compute std (ignore NaNs)
    def safe_std(x):
        return np.nanstd(x, ddof=1)
    
    return (
        safe_std(aucs),
        safe_std(prcs),
        safe_std(accs),
        safe_std(sens),
        safe_std(spes),
        safe_std(ppvs),
        safe_std(npvs),
    )